In [1]:
import  pandas as pd
import numpy as np
df=pd.read_csv("C:/Users/barku/OneDrive/Desktop/credit-risk-analysis/accepted_2007_to_2018Q4.csv")

C:\Users\barku\AppData\Local\Temp\ipykernel_12756\2443639921.py:3: DtypeWarning: Columns (0: id, 1: desc, 2: next_pymnt_d, 3: verification_status_joint, 4: sec_app_earliest_cr_line, 5: hardship_type, 6: hardship_reason, 7: hardship_status, 8: hardship_start_date, 9: hardship_end_date, 10: payment_plan_start_date, 11: hardship_loan_status, 12: debt_settlement_flag_date, 13: settlement_status, 14: settlement_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("C:/Users/barku/OneDrive/Desktop/credit-risk-analysis/accepted_2007_to_2018Q4.csv")


In [2]:
df

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2260696,88985880,NaN,40000.0,40000.0,40000.0,60 months,10.49,859.56,B,B3,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260697,88224441,NaN,24000.0,24000.0,24000.0,60 months,14.49,564.56,C,C4,...,NaN,NaN,Cash,Y,Mar-2019,ACTIVE,Mar-2019,10000.0,44.82,1.0
2260698,88215728,NaN,14000.0,14000.0,14000.0,60 months,14.49,329.33,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2260699,Total amount funded in policy code 1: 1465324575,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df.isnull().sum()

id                             0
member_id                2260701
loan_amnt                     33
funded_amnt                   33
funded_amnt_inv               33
                          ...   
settlement_status        2226455
settlement_date          2226455
settlement_amount        2226455
settlement_percentage    2226455
settlement_term          2226455
Length: 151, dtype: int64

In [4]:
#Check the shape first
df.shape

(2260701, 151)

In [5]:
# See null percentage for every column
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_pct = null_pct.sort_values(ascending=False)
print(null_pct[null_pct > 0])

member_id                                     100.00
orig_projected_additional_accrued_interest     99.62
hardship_reason                                99.52
hardship_payoff_balance_amount                 99.52
hardship_last_payment_amount                   99.52
                                               ...  
dti                                             0.08
pub_rec_bankruptcies                            0.06
tax_liens                                       0.01
collections_12_mths_ex_med                      0.01
chargeoff_within_12_mths                        0.01
Length: 102, dtype: float64


In [6]:
# Drop columns with more than 40% nulls
# Drop columns where more than 40% values are missing
threshold = 40
cols_to_drop = null_pct[null_pct > threshold].index.tolist()

print("Columns to drop:", len(cols_to_drop))
print(cols_to_drop)

df = df.drop(columns=cols_to_drop)
print("\nShape after dropping:", df.shape)

Columns to drop: 46
['member_id', 'orig_projected_additional_accrued_interest', 'hardship_reason', 'hardship_payoff_balance_amount', 'hardship_last_payment_amount', 'payment_plan_start_date', 'hardship_type', 'hardship_status', 'hardship_start_date', 'deferral_term', 'hardship_amount', 'hardship_dpd', 'hardship_loan_status', 'hardship_length', 'hardship_end_date', 'settlement_status', 'debt_settlement_flag_date', 'settlement_term', 'settlement_percentage', 'settlement_date', 'settlement_amount', 'sec_app_mths_since_last_major_derog', 'sec_app_revol_util', 'sec_app_num_rev_accts', 'sec_app_earliest_cr_line', 'sec_app_mort_acc', 'sec_app_open_acc', 'revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_open_act_il', 'sec_app_inq_last_6mths', 'sec_app_fico_range_high', 'sec_app_collections_12_mths_ex_med', 'sec_app_chargeoff_within_12_mths', 'verification_status_joint', 'annual_inc_joint', 'dti_joint', 'desc', 'mths_since_last_record', 'mths_since_recent_bc_dlq', 'mths_since_last_major_der

In [7]:
#  Keep only useful columns for credit risk

# Select only relevant columns for our analysis
useful_cols = [
    'loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment',
    'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc',
    'verification_status', 'loan_status', 'purpose', 'dti',
    'delinq_2yrs', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util',
    'total_acc', 'initial_list_status', 'application_type',
    'mort_acc', 'pub_rec_bankruptcies'
]

# Keep only columns that exist in our dataframe
useful_cols = [col for col in useful_cols if col in df.columns]
df = df[useful_cols]

print("Final columns:", df.shape)
print(df.columns.tolist())

Final columns: (2260701, 24)
['loan_amnt', 'funded_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'loan_status', 'purpose', 'dti', 'delinq_2yrs', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'application_type', 'mort_acc', 'pub_rec_bankruptcies']


In [8]:
#Check target variable
print(df['loan_status'].value_counts())
print("\n%")
print((df['loan_status'].value_counts() / len(df) * 100).round(2))

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

%
loan_status
Fully Paid                                             47.63
Current                                                38.85
Charged Off                                            11.88
Late (31-120 days)                                      0.95
In Grace Period                                         0.37
Late (16-30 days)                                       0.19
Does not meet 

In [9]:
# Filter only relevant loan statuses
# Keep only Fully Paid and Charged Off (our binary target)
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])]

print("Shape after filtering:", df.shape)
print(df['loan_status'].value_counts())

Shape after filtering: (1345310, 24)
loan_status
Fully Paid     1076751
Charged Off     268559
Name: count, dtype: int64


In [10]:
# Handle remaining nulls
# Fill numeric nulls with median
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical nulls with mode
cat_cols = df.select_dtypes(include='object').columns
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

print("Nulls remaining:", df.isnull().sum().sum())
print("Final shape:", df.shape)

C:\Users\barku\AppData\Local\Temp\ipykernel_12756\1826828680.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


Nulls remaining: 0
Final shape: (1345310, 24)


In [11]:
# Create target variable
# Convert loan_status to binary — 1 = Default, 0 = Fully Paid
df['target'] = (df['loan_status'] == 'Charged Off').astype(int)

print("Target distribution:")
print(df['target'].value_counts())
print("\n%")
print((df['target'].value_counts() / len(df) * 100).round(2))

# Drop original loan_status column
df = df.drop(columns=['loan_status'])

Target distribution:
target
0    1076751
1     268559
Name: count, dtype: int64

%
target
0    80.04
1    19.96
Name: count, dtype: float64


In [12]:
# Clean messy columns
# Fix int_rate — remove % sign if present
df['int_rate'] = df['int_rate'].astype(str).str.replace('%', '').astype(float)

# Fix term — extract number from '36 months' or '60 months'
df['term'] = df['term'].astype(str).str.extract('(\d+)').astype(float)

# Fix emp_length — extract number of years
df['emp_length'] = df['emp_length'].astype(str).str.extract('(\d+)')
df['emp_length'] = pd.to_numeric(df['emp_length'], errors='coerce')
df['emp_length'] = df['emp_length'].fillna(0)

print("Cleaned columns ✓")
df[['int_rate', 'term', 'emp_length']].head()

Cleaned columns ✓


,int_rate,term,emp_length
0,13.99,36.0,10
1,11.99,36.0,10
2,10.78,60.0,10
4,22.45,60.0,3
5,13.44,36.0,4


In [13]:
#  Encode categorical columns

from sklearn.preprocessing import LabelEncoder

cat_cols = df.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", cat_cols)

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print("Encoding done ✓")
df.head()

C:\Users\barku\AppData\Local\Temp\ipykernel_12756\2386799366.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns.tolist()


Categorical columns: ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose', 'initial_list_status', 'application_type']
Encoding done ✓


,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,...,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,target
0,3600.0,3600.0,36.0,13.99,123.03,2,13,10,1,55000.0,...,7.0,0.0,2765.0,29.7,13.0,1,0,1.0,0.0,0
1,24700.0,24700.0,36.0,11.99,820.28,2,10,10,1,65000.0,...,22.0,0.0,21470.0,19.2,38.0,1,0,4.0,0.0,0
2,20000.0,20000.0,60.0,10.78,432.66,1,8,10,1,63000.0,...,6.0,0.0,7869.0,56.2,18.0,1,1,5.0,0.0,0
4,10400.0,10400.0,60.0,22.45,289.91,5,25,3,1,104433.0,...,12.0,0.0,21929.0,64.5,35.0,1,0,6.0,0.0,0
5,11950.0,11950.0,36.0,13.44,405.18,2,12,4,5,34000.0,...,5.0,0.0,8822.0,68.4,6.0,1,0,0.0,0.0,0


In [14]:
df.to_csv("credit_risk_cleaned.csv", index=False)
print("Saved! ✓")
print("Final shape:", df.shape)
print("Columns:", df.columns.tolist())